In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import os
os.environ["GEMINI_API_KEY"] = "your_actual_key_here"

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

C:\Users\pranj\AppData\Local\Temp\ipykernel_21100\159965630.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader  = PyPDFLoader("./Data/medical_report.pdf")
docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data = splitter.split_documents(docs)

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [6]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    
    return context

In [7]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [8]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [9]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [14]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user","content":query}]})

Tool Called:  patient name doctor name
Tool Called:  Dr. Doctor Patient Name
Tool Called:  Dr. MD Pathologist Consultant


In [15]:
response

{'messages': [HumanMessage(content='What is the name of patient, and what is the name of Doctors', additional_kwargs={}, response_metadata={}, id='9a761ef1-2ae5-4788-a8b5-aebb1599af32'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'retriever_tool', 'arguments': '{"query": "patient name doctor name"}'}, '__gemini_function_call_thought_signatures__': {'call_1122198': 'ErUCCrICARFNMg8lWkVp+LIsYz1gvN1b6b6K2nXNKHqCJJDnKqHAy/n9MelK8EEG1LW08wZCslx4aDYrSb1MBAcoeqRhJEMb9GqN33WP9Hu4ej7LzSyIEo4vA2TvQ1do0jofIcv0b5rHPrwP6IeGZ4pTU15HspFB2fFk0iIvU4vJeSPZQRchT2k7Adume3+22+X+5tiZrlKE9a0ILK/wb7KttvwXj6ZDK2PtTr0KtXRcCd/pMQa/nxDOnpQ88Hq8NSL+yERWALUTjeCBmjvpTLBshEuT3373BWo1kTlILvtnG997YKKT78R6np8xqFe6icqBk00MXImOzvnTKEK9Vqqe8j5OsiijY1T1k7B7BcvNplUnjxYuo+8TiFjbQMmjDLU/FgYvHxGZVleiMKij/5SgheF+wttK'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0af8b-c917-7dc3-8f82-66ec0d99753a-0

In [17]:
result = response["messages"][-1].content

In [ ]:
print(result)

[{'type': 'text', 'text': 'Based on the medical report:\n\n* **Patient Name:** Ms. NIKITA CHUDHARY\n* **Doctor Name (Referring Physician):** DR NITIN NAHAR', 'extras': {'signature': 'EtQDCtEDARFNMg9kL83EzbAOf3OxbAMtcSOIvEuR+S9lolAueZfbRfR9ew2picc8h50c8MipM9VMrjtvUdBkNr5zbuhkYKEfmm90ruW/Ry6hRAVY1rSjqYm95x6lfqt6WwPuUbKHeCMK8ECTSDijsGRHiCV4onk8vxMVjaW3BuV8r/JOXYY27RUQS68PECE7wQ8yg4X7WBNO0QRKu32oWXMMYuyLDznNVMeGcSpkfhY/Es6xH3alPkYlM9Eg7r0Ds1qg+sNe4amMaMuaW5JRM6zQhubhq5D0lUrvMax3zLqhHD31iv+h6yMkdbX68RmAQM9Q1Omt3WOckWMaltHreXArIygJ0Jx4WIaRW+qU6iXeoZHd8pWFki+wmyg5D7KuJkmixJEiMJ+JN8l70iu5Pg2Y2CuEgbbBXBhSwJ/9k+NwgNcZerIC4wbkOXaK7IGeDr3iGqUfn71D3Hs0XWNqUzGDabZlEYgtfvekENoP1PQ7JSv4lfNglDailOQTfGjKkX5VBf+9Hhu3up2uogDXODVOaTMDSSQFri9sOmr+w6q/9qEEiw2VAYFKiaZSKIxut1yMnwuBsdCS0FW9N5bHKBu9EoCjSu2clVZMEMzdzzfaiB8iXKHt'}}]
